In [1]:
!pip install -q timm

In [1]:
import time
import gc
import numpy as np
import pandas as pd
import torch
import timm

assert torch.cuda.is_available(), "Switch your Colab runtime to T4 GPU first!"
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

resolutions = [128, 256, 384]
BS = 16

models_res = {
    "MobileNetV2": lambda: timm.create_model("mobilenetv2_100", pretrained=True).eval(),
    "MobileViT-S": lambda: timm.create_model("mobilevit_s", pretrained=True).eval(),
}

configs = [
    ("GPU", "FP32"),
    ("GPU", "FP16"),
    ("CPU", "FP32"),
]

rows = []

for dev_name, prec in configs:
    device = torch.device("cuda" if dev_name == "GPU" else "cpu")
    use_fp16 = (dev_name == "GPU" and prec == "FP16")

    for model_name, builder in models_res.items():
        print(f"\n=== {dev_name} {prec} | {model_name} (BS={BS}) ===")
        model = builder().to(device)

        for res in resolutions:
            warmups = 5 if dev_name == "GPU" else 1
            repeats = 20 if dev_name == "GPU" else (5 if res <= 256 else 3)

            try:
                x = torch.randn(BS, 3, res, res, device=device)
                if dev_name == "GPU":
                    torch.cuda.reset_peak_memory_stats()

                # Warm-up
                with torch.no_grad():
                    for _ in range(warmups):
                        with torch.amp.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
                            _ = model(x)
                    if dev_name == "GPU":
                        torch.cuda.synchronize()

                # Steady-state timing
                times = []
                with torch.no_grad():
                    for _ in range(repeats):
                        t0 = time.perf_counter()
                        with torch.amp.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
                            _ = model(x)
                        if dev_name == "GPU":
                            torch.cuda.synchronize()
                        t1 = time.perf_counter()
                        times.append((t1 - t0) * 1000.0)

                med_ms = float(np.median(times))
                p95_ms = float(np.percentile(times, 95))
                tput = BS / (med_ms / 1000.0)
                peak_mem = (
                    torch.cuda.max_memory_allocated() / (1024 ** 2)
                    if dev_name == "GPU" else np.nan
                )

                rows.append({
                    "Device": dev_name,
                    "Precision": prec,
                    "Model": model_name,
                    "Batch_Size": BS,
                    "Resolution": res,
                    "Pixels": res * res,
                    "Steady_Median_ms": round(med_ms, 3),
                    "Steady_P95_ms": round(p95_ms, 3),
                    "Throughput_samples_s": round(tput, 2),
                    "Peak_GPU_Mem_MB": round(peak_mem, 2) if not np.isnan(peak_mem) else np.nan,
                    "Status": "OK",
                })
                print(
                    f"{res}x{res} | Steady: {med_ms:8.2f} ms | "
                    f"Tput: {tput:7.1f} img/s | Mem: {peak_mem if not np.isnan(peak_mem) else 0:6.1f} MB"
                )
            except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
                if "out of memory" in str(e).lower():
                    rows.append({
                        "Device": dev_name, "Precision": prec, "Model": model_name,
                        "Batch_Size": BS, "Resolution": res, "Pixels": res * res,
                        "Steady_Median_ms": np.nan, "Steady_P95_ms": np.nan,
                        "Throughput_samples_s": np.nan, "Peak_GPU_Mem_MB": np.nan,
                        "Status": "OOM",
                    })
                    print(f"{res}x{res} | OOM!")
                else:
                    raise
            finally:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

res_cg_df = pd.DataFrame(rows)
res_cg_df.to_csv("resolution_sweep_cpu_gpu.csv", index=False)
print("\nSaved resolution_sweep_cpu_gpu.csv")


=== GPU FP32 | MobileNetV2 (BS=16) ===


model.safetensors: reconstructing file:   0%|          |  0.00B / 14.2MB            

model.safetensors: downloading bytes:           |  0.00B            

128x128 | Steady:     7.58 ms | Tput:  2112.1 img/s | Mem:   84.7 MB
256x256 | Steady:    17.30 ms | Tput:   924.6 img/s | Mem:  273.7 MB
384x384 | Steady:    38.68 ms | Tput:   413.6 img/s | Mem:  589.7 MB

=== GPU FP32 | MobileViT-S (BS=16) ===


model.safetensors: reconstructing file:   0%|          |  0.00B / 22.4MB            

model.safetensors: downloading bytes:           |  0.00B            

128x128 | Steady:    17.33 ms | Tput:   923.1 img/s | Mem:  108.5 MB
256x256 | Steady:    40.91 ms | Tput:   391.1 img/s | Mem:  345.5 MB
384x384 | Steady:    98.90 ms | Tput:   161.8 img/s | Mem:  741.5 MB

=== GPU FP16 | MobileNetV2 (BS=16) ===
128x128 | Steady:     7.96 ms | Tput:  2009.4 img/s | Mem:   54.7 MB
256x256 | Steady:     9.51 ms | Tput:  1682.0 img/s | Mem:  153.7 MB
384x384 | Steady:    19.33 ms | Tput:   827.9 img/s | Mem:  319.7 MB

=== GPU FP16 | MobileViT-S (BS=16) ===
128x128 | Steady:    15.04 ms | Tput:  1063.6 img/s | Mem:   70.5 MB
256x256 | Steady:    27.19 ms | Tput:   588.4 img/s | Mem:  274.0 MB
384x384 | Steady:    76.08 ms | Tput:   210.3 img/s | Mem:  982.1 MB

=== CPU FP32 | MobileNetV2 (BS=16) ===
128x128 | Steady:   183.65 ms | Tput:    87.1 img/s | Mem:    0.0 MB
256x256 | Steady:  1234.41 ms | Tput:    13.0 img/s | Mem:    0.0 MB
384x384 | Steady:  2722.84 ms | Tput:     5.9 img/s | Mem:    0.0 MB

=== CPU FP32 | MobileViT-S (BS=16) ===
128x128 | St

In [2]:
from torch.profiler import profile, ProfilerActivity

prof_rows = []
x_prof = torch.randn(1, 3, 256, 256)

for model_name, builder in models_res.items():
    print(f"\n================ CPU Profiler: {model_name} (256x256, BS=1) ================")
    model = builder().to("cpu")

    # Warm-up
    with torch.no_grad():
        _ = model(x_prof)

    # Profile 3 forward passes
    with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
        with torch.no_grad():
            for _ in range(3):
                _ = model(x_prof)

    print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=10))

    total_self_us = sum(evt.self_cpu_time_total for evt in prof.key_averages())
    top_events = sorted(
        prof.key_averages(), key=lambda e: e.self_cpu_time_total, reverse=True
    )[:10]

    for evt in top_events:
        prof_rows.append({
            "Model": model_name,
            "Operator": evt.key,
            "Calls": evt.count,
            "Self_CPU_ms": round(evt.self_cpu_time_total / 1000.0, 3),
            "Self_CPU_%": round(100.0 * evt.self_cpu_time_total / total_self_us, 2),
            "Total_CPU_ms": round(evt.cpu_time_total / 1000.0, 3),
        })

    del model
    gc.collect()

prof_df = pd.DataFrame(prof_rows)
prof_df.to_csv("cpu_profiler_ops.csv", index=False)
print("\nSaved cpu_profiler_ops.csv")


================ CPU Profiler: MobileNetV2 (256x256, BS=1) ================
--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                            Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
        aten::mkldnn_convolution        33.04%      30.515ms        33.80%      31.214ms     578.042us            54  
      aten::_slow_conv2d_forward        28.87%      26.663ms        29.74%      27.465ms     269.267us           102  
         aten::native_batch_norm        17.35%      16.026ms        18.67%      17.238ms     110.500us           156  
                     aten::clamp         6.22%       5.746ms         6.29%       5.806ms      55.292us           105  
    aten::_batch_norm_impl_index         2.21%       2.038ms        21.20%      19.582ms  

In [2]:
import time
import gc
import numpy as np
import pandas as pd
import torch
import timm

torch.manual_seed(42)

USE_XLA = False
try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    USE_XLA = True
except Exception:
    device = torch.device("tpu")

print(f"Active TPU device: {device}")

def sync_tpu(tensor=None):
    if USE_XLA:
        xm.mark_step()
        xm.wait_device_ops()
    elif tensor is not None:
        _ = tensor.cpu()

resolutions = [128, 256, 384]
BS = 16
WARMUPS = 3
REPEATS = 15

models_res = {
    "MobileNetV2": lambda: timm.create_model("mobilenetv2_100", pretrained=True).eval(),
    "MobileViT-S": lambda: timm.create_model("mobilevit_s", pretrained=True).eval(),
}

res_rows = []

for name, builder in models_res.items():
    print(f"\n=== TPU Resolution Sweep: {name} (BS={BS}, BF16) ===")
    model = builder().to(device=device, dtype=torch.bfloat16)

    for res in resolutions:
        try:
            x_dev = torch.randn(BS, 3, res, res, dtype=torch.bfloat16).to(device)
            sync_tpu(x_dev)

            # 1st pass (includes XLA compile for this resolution)
            with torch.no_grad():
                t0 = time.perf_counter()
                out = model(x_dev)
                sync_tpu(out)
                t1 = time.perf_counter()
            compile_ms = (t1 - t0) * 1000.0

            # Warm-up
            with torch.no_grad():
                for _ in range(WARMUPS):
                    out = model(x_dev)
                    sync_tpu(out)

            # Steady-state latency
            times = []
            with torch.no_grad():
                for _ in range(REPEATS):
                    t0 = time.perf_counter()
                    out = model(x_dev)
                    sync_tpu(out)
                    t1 = time.perf_counter()
                    times.append((t1 - t0) * 1000.0)

            med_ms = float(np.median(times))
            p95_ms = float(np.percentile(times, 95))
            tput = BS / (med_ms / 1000.0)

            res_rows.append({
                "Device": "TPU",
                "Precision": "BF16",
                "Model": name,
                "Batch_Size": BS,
                "Resolution": res,
                "Pixels": res * res,
                "Compile_Plus_First_ms": round(compile_ms, 2),
                "Steady_Median_ms": round(med_ms, 3),
                "Steady_P95_ms": round(p95_ms, 3),
                "Throughput_samples_s": round(tput, 2),
                "Status": "OK",
            })
            print(
                f"{name:12} | {res}x{res} | Compile+1st: {compile_ms:8.1f} ms | "
                f"Steady: {med_ms:7.2f} ms | Tput: {tput:7.1f} img/s"
            )
        except RuntimeError as e:
            if "out of memory" in str(e).lower() or "resource_exhausted" in str(e).lower():
                res_rows.append({
                    "Device": "TPU", "Precision": "BF16", "Model": name,
                    "Batch_Size": BS, "Resolution": res, "Pixels": res * res,
                    "Compile_Plus_First_ms": np.nan, "Steady_Median_ms": np.nan,
                    "Steady_P95_ms": np.nan, "Throughput_samples_s": np.nan,
                    "Status": "OOM",
                })
                print(f"{name:12} | {res}x{res} | OOM!")
            else:
                raise

    del model
    gc.collect()

res_tpu_df = pd.DataFrame(res_rows)
res_tpu_df.to_csv("resolution_sweep_tpu.csv", index=False)
print("\nSaved resolution_sweep_tpu.csv")

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


Active TPU device: tpu

=== TPU Resolution Sweep: MobileNetV2 (BS=16, BF16) ===


model.safetensors: reconstructing file:   0%|          |  0.00B / 14.2MB            

model.safetensors: downloading bytes:           |  0.00B            

MobileNetV2  | 128x128 | Compile+1st:   4694.0 ms | Steady:   13.03 ms | Tput:  1227.8 img/s
MobileNetV2  | 256x256 | Compile+1st:   8167.9 ms | Steady:   12.81 ms | Tput:  1248.9 img/s
MobileNetV2  | 384x384 | Compile+1st:   9301.2 ms | Steady:   18.65 ms | Tput:   857.8 img/s

=== TPU Resolution Sweep: MobileViT-S (BS=16, BF16) ===


model.safetensors: reconstructing file:   0%|          |  0.00B / 22.4MB            

model.safetensors: downloading bytes:           |  0.00B            

MobileViT-S  | 128x128 | Compile+1st:  11195.9 ms | Steady:   19.62 ms | Tput:   815.4 img/s
MobileViT-S  | 256x256 | Compile+1st:  13255.8 ms | Steady:   21.04 ms | Tput:   760.5 img/s
MobileViT-S  | 384x384 | Compile+1st:  14971.4 ms | Steady:   35.57 ms | Tput:   449.9 img/s

Saved resolution_sweep_tpu.csv
